# Data and annotations

Run the CPU fixture without datasets or model weights. Every download, video import and SAM2 job is explicitly configured below. The current default is 100 public RELLIS frames and automatic `rellis_material_v1` references. Phone clips are pending optional inputs. `configs/data/legacy_phone_100.json` retains the original 70-public/30-phone recipe in a separate run.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [path for base in (cwd, *cwd.parents) for path in (base, base / "VLM_evaluation")]
REPO = next((p for p in candidates if (p / "docs/interfaces.md").is_file()), None)
if REPO is None:
    raise RuntimeError("Start this notebook inside VLM_evaluation or its repository checkout.")
sys.path.insert(0, str(REPO / "src"))

from traversability_data import (
    resolve_roots, read_json, read_jsonl, resource_catalog, download_resource, import_archive,
    import_rellis, import_tum, import_phone_video, prepare_run, import_sam_masks,
    load_sam2_generator, generate_sam2_regions, download_sam2_checkpoint,
    freeze_selection, derive_references, annotation_viewer, show_region, save_annotation,
    coverage_summary, export_run, create_cpu_fixture,
)

CONFIG_PATH = REPO / "configs/data/default.json"
SETTINGS = read_json(CONFIG_PATH)
ROOTS = resolve_roots(repo_root=REPO, data_root=None, run_root=None)
RUN_NAME = "public-rellis-v1"  # Use a different name when changing recipe, policy or inputs.
RUN_DIR = ROOTS.run_root / RUN_NAME

RUN_FIXTURE = True
DOWNLOAD_KEYS = []  # Explicit per-resource opt-in: rellis_rgb, rellis_semantic, tum_floor, tum_walking_xyz.
IMPORT_ARCHIVES = {}  # e.g. {"rellis_rgb": Path("/external/completed.zip")}
RELLIS_SOURCE_DIR = None  # Existing publisher directory; None uses imported raw/ directories.
TUM_SOURCE_DIRS = {}  # {"freiburg1_floor": extracted_directory, "freiburg3_walking_xyz": extracted_directory}
PHONE_CLIPS = []  # [{"video_path": Path(...), "location": "A", "clip_id": "clip1"}]; five keyframes per clip.
MASK_TARGET = "public"  # Set to "phone" for imported phone masks and annotations.
MASK_IMPORT_JSONL = None  # External SAM producer records; paths relative to ROOTS.data_root or absolute import paths.
DOWNLOAD_SAM2 = False
RUN_SAM2 = False
SAM2_CHECKPOINT = ROOTS.data_root / "checkpoints/sam2.1_hiera_tiny.pt"
SAM2_REVISION = None  # Optional declared installed source commit; actual provenance is also recorded.
OPEN_VIEWER = False
EXPORT_DIRECTORY = None  # Choose a new snapshot directory under the run root.

print({"data_root": str(ROOTS.data_root), "run_root": str(ROOTS.run_root), "recipe": SETTINGS["recipe"]})

## CPU fixture

Synthetic RGB and masks demonstrate aligned import, the 2-small/1-medium/2-large freeze, one persisted annotation and safe resumption. They never count as real coverage.

In [ ]:
if RUN_FIXTURE:
    FIXTURE_DATA = ROOTS.data_root / "fixtures"
    FIXTURE_RUN = ROOTS.run_root / "cpu-data-fixture"
    fixture = create_cpu_fixture(FIXTURE_DATA, FIXTURE_RUN)
    print(fixture["coverage"])
    # Optional display: show_region(FIXTURE_RUN, FIXTURE_DATA, fixture["demonstration_region_id"])

## Download or import public sources

The resource list is read from the fixed manifest. Set only the resource keys you want to download. RELLIS RGB is approximately 11 GB; Drive access and quotas may prevent transfer. A completed manual download from the exact link can be supplied in `IMPORT_ARCHIVES`. Downloads use temporary files and preserve completed data.

In [ ]:
RESOURCES = resource_catalog(REPO / "research/traversability/data_manifest.json")
for key, resource in RESOURCES.items():
    print(key, resource.get("publisher_size", resource.get("expected_bytes")), resource["url"])

completed_archives = dict(IMPORT_ARCHIVES)
for key in DOWNLOAD_KEYS:
    if key not in RESOURCES:
        raise ValueError(f"Unknown resource: {key}")
    completed_archives[key] = download_resource(RESOURCES[key], ROOTS.data_root, allow_download=True)
for key, archive in completed_archives.items():
    print(import_archive(archive, ROOTS.data_root, key))

In [ ]:
imported = import_rellis(
    ROOTS.data_root, source_dir=RELLIS_SOURCE_DIR,
    recipe=SETTINGS.get("rellis_recipe"), allow_partial=True,
)
prepare_run(
    RUN_DIR, imported["frames"], ROOTS.data_root,
    metadata={key: SETTINGS[key] for key in ("recipe", "expected_frames", "robot_profile_id", "robot_policy")},
    semantic_aids=imported["semantic_aids"],
)
if imported["missing"]:
    print("Preparation gaps (no substitution or fabricated frames):", imported["missing"])
print(coverage_summary(RUN_DIR))

## Supplementary TUM and future phone clips

TUM RGB timestamps come from `rgb.txt`; its runs do not fill the public keyframe recipe. Phone timestamps come from presentation timestamps, with decoded orientation preserved. For the legacy recipe, import two five-keyframe clips at each location into the legacy run; for the current default, use a separate phone run.

In [ ]:
for sequence, directory in TUM_SOURCE_DIRS.items():
    frames = import_tum(ROOTS.data_root, directory, sequence)
    tum_run = ROOTS.run_root / f"tum-{sequence}"
    prepare_run(tum_run, frames, ROOTS.data_root, metadata={"recipe": "supplementary_tum", "expected_frames": {}})
    print(coverage_summary(tum_run))

phone_run = RUN_DIR if SETTINGS["recipe"] == "legacy_phone" else ROOTS.run_root / "phone-extension-v1"
for clip in PHONE_CLIPS:
    frames = import_phone_video(ROOTS.data_root, clip["video_path"], clip["location"], clip["clip_id"], keyframes=5)
    prepare_run(phone_run, frames, ROOTS.data_root,
                metadata=None if phone_run == RUN_DIR else {
                    "recipe": "phone_extension", "expected_frames": {"development": 10, "test": 20},
                    "robot_profile_id": "small_wheeled_v1",
                    "robot_policy": read_json(REPO / "configs/data/legacy_phone_100.json")["robot_policy"],
                })
if not PHONE_CLIPS:
    print("Phone clips: pending; no phone hazard results or fabricated frames.")

## Generate or import SAM masks

Import producer JSONL rows containing `region_id`, `frame_id`, `mask_path` and optional `planning_relevant`, `image_sha256` and generator provenance. Every mask must match its original RGB. Dataset semantic masks are reference aids, never SAM predictions.

SAM2 is optional: follow the [official installation instructions](https://github.com/facebookresearch/sam2/blob/main/INSTALL.md) in an isolated environment. Python ≥3.10, PyTorch ≥2.5.1 and torchvision ≥0.20.1 are required. Windows guidance recommends Ubuntu under WSL; `SAM2_BUILD_CUDA=0` omits the optional extension. The tiny checkpoint/config is a configurable starter, without a measured quality/resource claim.

In [ ]:
if MASK_TARGET not in {"public", "phone"}:
    raise ValueError("MASK_TARGET must be public or phone.")
ACTIVE_RUN_DIR = RUN_DIR if MASK_TARGET == "public" else phone_run
if not (ACTIVE_RUN_DIR / "frames.jsonl").is_file():
    raise FileNotFoundError("Phone clips are pending. Import PHONE_CLIPS before choosing MASK_TARGET='phone'.")
if DOWNLOAD_SAM2:
    download_sam2_checkpoint(SAM2_CHECKPOINT, model=SETTINGS.get("sam2_model", "tiny"), allow_download=True)

if MASK_IMPORT_JSONL is not None and RUN_SAM2:
    raise ValueError("Choose external-mask import or SAM2 generation for this run.")
if MASK_IMPORT_JSONL is not None:
    import_sam_masks(ACTIVE_RUN_DIR, ROOTS.data_root, read_jsonl(MASK_IMPORT_JSONL), mask_origin="external_sam")
if RUN_SAM2:
    generator = load_sam2_generator(
        SAM2_CHECKPOINT, model_config=SETTINGS.get("sam2_config", "configs/sam2.1/sam2.1_hiera_t.yaml"),
        device=SETTINGS.get("sam2_device", "cpu"), sam2_revision=SAM2_REVISION,
    )
    try:
        generate_sam2_regions(ACTIVE_RUN_DIR, ROOTS.data_root, generator)
    finally:
        del generator

selection = freeze_selection(ACTIVE_RUN_DIR, ROOTS.data_root, seed=SETTINGS.get("selection_seed", 0))
print({"frozen_frames": len(selection["frames"]), "frames_pending_masks": len(selection["pending_frame_ids"])})

## References, optional annotation and resume

Automatic RELLIS references apply the frozen material policy with 95% non-void coverage and class purity. Any avoided-pixel overlap prevents automatic traversable assignment. Missing aids, void and unresolved grass remain null/pending. Semantic IDs and reference fractions are stored outside VLM input records.

The viewer is optional and skips completed rows. Without widgets, it returns a function-based session with `pending_region_ids`, `show()` and `save()`. `unknown` is an explicit label; missing labels stay pending.

In [ ]:
ACTIVE_POLICY = read_json(ACTIVE_RUN_DIR / "metadata/data.json")["robot_profile_id"]
if ACTIVE_POLICY == "rellis_material_v1":
    derive_references(
        ACTIVE_RUN_DIR, ROOTS.data_root,
        coverage_threshold=SETTINGS.get("reference_nonvoid_threshold", 0.95),
        purity_threshold=SETTINGS.get("reference_class_purity_threshold", 0.95),
    )
print(coverage_summary(ACTIVE_RUN_DIR))
if OPEN_VIEWER:
    display(annotation_viewer(ACTIVE_RUN_DIR, ROOTS.data_root))

# Plain fallback:
# session = annotation_viewer(ACTIVE_RUN_DIR, ROOTS.data_root, use_widgets=False)
# session.pending_region_ids
# session.show()
# session.save("non_traversable", mask_quality="valid", hazard_type="cable")
#
# Rerun preparation/import/freeze/reference cells with the same settings to resume.
# Completed annotations, RGB and masks are preserved; changed frozen inputs require a new run.

## Export

The run already contains the fixed JSONL artifacts. An optional export makes a validated immutable snapshot; RGB and mask paths still resolve against the configured data root. Inference uses only `frames.jsonl` and selected SAM rows in `regions.jsonl`. Evaluation may read annotations and reference provenance.

In [ ]:
if EXPORT_DIRECTORY is not None:
    print(export_run(ACTIVE_RUN_DIR, ROOTS.data_root, EXPORT_DIRECTORY))
else:
    print("Artifacts:", ACTIVE_RUN_DIR)
print(coverage_summary(ACTIVE_RUN_DIR))